# Regression Tree

California housing, target `median_house_value`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

y = data_df['median_house_value'].to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()
feature_names = list(data_df.drop(columns=['median_house_value']).columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, shuffle=True, random_state=42
)

print(X_train.shape, X_test.shape)

## Baseline

In [ ]:
reg = DecisionTreeRegressor(random_state=42)
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R2:  {r2_score(y_test, y_pred):.4f}")
print(f"Depth: {reg.get_depth()}  Leaves: {reg.get_n_leaves()}")

## Depth sweep

In [ ]:
depths = list(range(1, 26))
train_r2, test_r2 = [], []

for d in depths:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_r2.append(r2_score(y_train, m.predict(X_train)))
    test_r2.append(r2_score(y_test, m.predict(X_test)))

plt.figure(figsize=(9, 5))
plt.plot(depths, train_r2, 'o-', label='train R2')
plt.plot(depths, test_r2, 's-', label='test R2')
plt.xlabel('max_depth')
plt.ylabel('R2')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

best_d = depths[int(np.argmax(test_r2))]
print(f"Best depth: {best_d}, test R2: {max(test_r2):.4f}")

## Splitting criterion

In [ ]:
for crit in ['squared_error', 'friedman_mse', 'absolute_error']:
    m = DecisionTreeRegressor(criterion=crit, max_depth=best_d, random_state=42)
    m.fit(X_train, y_train)
    te = r2_score(y_test, m.predict(X_test))
    mae = mean_absolute_error(y_test, m.predict(X_test))
    print(f"{crit:<16} R2={te:.4f}  MAE={mae:.2f}")

## Feature importance

In [ ]:
best = DecisionTreeRegressor(max_depth=best_d, random_state=42).fit(X_train, y_train)
importance = pd.Series(best.feature_importances_, index=feature_names).sort_values()

plt.figure(figsize=(9, 6))
importance.plot(kind='barh', color='steelblue')
plt.xlabel('importance')
plt.grid(alpha=0.3)
plt.show()

print(importance)

## Predicted vs actual

In [ ]:
y_pred = best.predict(X_test)

fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].scatter(y_test, y_pred, alpha=0.2, s=8)
axs[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axs[0].set_xlabel('actual')
axs[0].set_ylabel('predicted')

axs[1].hist(y_pred - y_test, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axs[1].axvline(0, color='red', linestyle='--')
axs[1].set_xlabel('error ($)')
axs[1].set_ylabel('frequency')

plt.tight_layout()
plt.show()

## Tree visualization (shallow)

In [ ]:
small = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, y_train)
plt.figure(figsize=(20, 10))
plot_tree(small, feature_names=feature_names, filled=True, rounded=True, fontsize=10)
plt.show()